# 02 -- Preprocessing Pipeline: Shin 2017 EEG+NIRS Dataset

Bandpass EEG (1-40 Hz), lowpass NIRS (0.2 Hz), extract sliding windows, save processed data.

In [1]:
import numpy as np
import scipy.io
from scipy.signal import butter, sosfiltfilt
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pickle
import os
from pathlib import Path

print('Imports OK')


Imports OK


## 1. Data Loading

In [2]:
def load_subject(subj, task, data_root):
    """
    Load EEG, NIRS, and markers for one subject and task.

    Returns dict with:
      'eeg': np.ndarray (T_eeg, 30) -- EEG in uV at 200 Hz
      'nirs_hbo': np.ndarray (T_nirs, 36) -- HbO at 10 Hz
      'nirs_hbr': np.ndarray (T_nirs, 36) -- HbR at 10 Hz
      'eeg_fs': float -- EEG sampling rate (should be 200)
      'nirs_fs': float -- NIRS sampling rate (should be 10)
      'eeg_mrk_times_ms': np.ndarray -- trial onset times in ms (EEG clock)
      'nirs_mrk_times_ms': np.ndarray -- trial onset times in ms (NIRS clock)
      'labels': np.ndarray of int -- 1=task (VF), 0=baseline (BL)
      'class_names': list of str
    """
    data_root = Path(data_root)
    subj_dir = data_root / subj

    var = f'cnt_{task}'
    mrk_var = f'mrk_{task}'

    # Load EEG
    eeg_path = subj_dir / 'EEG' / f'cnt_{task}.mat'
    eeg_mat = scipy.io.loadmat(str(eeg_path), squeeze_me=True, struct_as_record=False)
    cnt_eeg = eeg_mat[var]
    eeg_data = cnt_eeg.x  # (T, C)
    eeg_fs = float(cnt_eeg.fs)

    # Load EEG markers
    eeg_mrk_path = subj_dir / 'EEG' / f'mrk_{task}.mat'
    eeg_mrk_mat = scipy.io.loadmat(str(eeg_mrk_path), squeeze_me=True, struct_as_record=False)
    mrk_eeg = eeg_mrk_mat[mrk_var]
    eeg_mrk_times = np.atleast_1d(mrk_eeg.time).astype(float)

    # Load NIRS
    nirs_path = subj_dir / 'NIRS' / f'cnt_{task}.mat'
    nirs_mat = scipy.io.loadmat(str(nirs_path), squeeze_me=True, struct_as_record=False)
    cnt_nirs = nirs_mat[var]
    nirs_hbo = cnt_nirs.oxy.x    # (T, 36)
    nirs_hbr = cnt_nirs.deoxy.x  # (T, 36)
    nirs_fs = float(cnt_nirs.oxy.fs)

    # Load NIRS markers
    nirs_mrk_path = subj_dir / 'NIRS' / f'mrk_{task}.mat'
    nirs_mrk_mat = scipy.io.loadmat(str(nirs_mrk_path), squeeze_me=True, struct_as_record=False)
    mrk_nirs = nirs_mrk_mat[mrk_var]
    nirs_mrk_times = np.atleast_1d(mrk_nirs.time).astype(float)

    # Labels: mrk.y is 2D one-hot (2, n_trials), argmax gives 0=BL, 1=VF
    y = mrk_eeg.y
    if y.ndim == 1:
        labels = np.atleast_1d(y).astype(int)
    else:
        labels = np.argmax(y, axis=0).astype(int)

    # class names
    class_names = list(mrk_eeg.className)

    return {
        'eeg': eeg_data,
        'nirs_hbo': nirs_hbo,
        'nirs_hbr': nirs_hbr,
        'eeg_fs': eeg_fs,
        'nirs_fs': nirs_fs,
        'eeg_mrk_times_ms': eeg_mrk_times,
        'nirs_mrk_times_ms': nirs_mrk_times,
        'labels': labels,
        'class_names': class_names,
    }

print('load_subject defined')


load_subject defined


## 2. EEG Bandpass Filter (1-40 Hz)

In [3]:
def bandpass_eeg(eeg, fs, low=1.0, high=40.0, order=6):
    """Apply zero-phase bandpass filter to EEG. Input: (T, C), Output: (T, C)"""
    sos = butter(order, [low, high], btype='bandpass', fs=fs, output='sos')
    try:
        return sosfiltfilt(sos, eeg, axis=0)
    except Exception as e:
        print(f'  Warning: bandpass_eeg failed ({e}), applying per-channel')
        out = np.zeros_like(eeg)
        for c in range(eeg.shape[1]):
            try:
                out[:, c] = sosfiltfilt(sos, eeg[:, c])
            except Exception as e2:
                print(f'    Channel {c} skipped: {e2}')
                out[:, c] = eeg[:, c]
        return out

print('bandpass_eeg defined')


bandpass_eeg defined


## 3. NIRS Low-pass Filter (0.2 Hz)

In [4]:
def lowpass_nirs(nirs, fs, cutoff=0.2, order=6):
    """Apply zero-phase low-pass filter to NIRS. Input: (T, C), Output: (T, C)"""
    sos = butter(order, cutoff, btype='low', fs=fs, output='sos')
    try:
        return sosfiltfilt(sos, nirs, axis=0)
    except Exception as e:
        print(f'  Warning: lowpass_nirs failed ({e}), applying per-channel')
        out = np.zeros_like(nirs)
        for c in range(nirs.shape[1]):
            try:
                out[:, c] = sosfiltfilt(sos, nirs[:, c])
            except Exception as e2:
                print(f'    Channel {c} skipped: {e2}')
                out[:, c] = nirs[:, c]
        return out

print('lowpass_nirs defined')


lowpass_nirs defined


## 4. Sliding Window Extraction

In [5]:
def extract_windows(data, task='vf', window_sec=5, step_sec=1):
    """
    Extract sliding windows from each trial.

    For VF task: task duration = 10s, rest = 13-15s
    Positive pairs: (eeg_window, nirs_window) from same trial, same time offset

    EEG window shape:  (window_sec * eeg_fs, 30)  = (1000, 30)
    NIRS window shape: (window_sec * nirs_fs, 72)  = (50, 72)  [HbO + HbR concatenated]

    Returns:
      eeg_windows:  np.ndarray (N, 30, 1000)  -- transposed for model input (C, T)
      nirs_windows: np.ndarray (N, 72, 50)    -- transposed for model input (C, T)
      labels:       np.ndarray (N,)
    """
    eeg_fs = data['eeg_fs']
    nirs_fs = data['nirs_fs']
    eeg_win = int(window_sec * eeg_fs)    # 1000
    nirs_win = int(window_sec * nirs_fs)  # 50
    eeg_step = int(step_sec * eeg_fs)     # 200
    nirs_step = int(step_sec * nirs_fs)   # 10

    eeg_data = data['eeg']         # (T_eeg, 30)
    nirs_hbo = data['nirs_hbo']   # (T_nirs, 36)
    nirs_hbr = data['nirs_hbr']   # (T_nirs, 36)
    nirs_data = np.concatenate([nirs_hbo, nirs_hbr], axis=1)  # (T_nirs, 72)

    eeg_mrk_ms = data['eeg_mrk_times_ms']
    nirs_mrk_ms = data['nirs_mrk_times_ms']
    labels_raw = data['labels']  # 1=VF, 0=BL

    # trial duration in samples
    task_dur_eeg = int(10 * eeg_fs)    # 10s task period
    task_dur_nirs = int(10 * nirs_fs)

    eeg_wins, nirs_wins, lbls = [], [], []

    for i, (eeg_t_ms, nirs_t_ms, lbl) in enumerate(zip(eeg_mrk_ms, nirs_mrk_ms, labels_raw)):
        eeg_onset = int(eeg_t_ms * eeg_fs / 1000)
        nirs_onset = int(nirs_t_ms * nirs_fs / 1000)

        # slide window across trial
        t_eeg = eeg_onset
        t_nirs = nirs_onset
        while (t_eeg + eeg_win <= eeg_onset + task_dur_eeg and
               t_eeg + eeg_win <= eeg_data.shape[0] and
               t_nirs + nirs_win <= nirs_onset + task_dur_nirs and
               t_nirs + nirs_win <= nirs_data.shape[0]):
            eeg_wins.append(eeg_data[t_eeg:t_eeg + eeg_win, :].T)     # (30, 1000)
            nirs_wins.append(nirs_data[t_nirs:t_nirs + nirs_win, :].T) # (72, 50)
            lbls.append(lbl)
            t_eeg += eeg_step
            t_nirs += nirs_step

    return (np.array(eeg_wins, dtype=np.float32),
            np.array(nirs_wins, dtype=np.float32),
            np.array(lbls, dtype=np.int64))

print('extract_windows defined')

extract_windows defined


## 5. Run Full Pipeline on VP001-VP005

In [6]:
os.makedirs('../data/processed', exist_ok=True)
DATA_ROOT = Path('../data/raw')
SUBJECTS = ['VP001', 'VP002', 'VP003', 'VP004', 'VP005']

all_data = {}
for subj in SUBJECTS:
    print(f'\n--- {subj} ---')
    try:
        data = load_subject(subj, 'vf', DATA_ROOT)
        print(f"  EEG shape: {data['eeg'].shape}, fs={data['eeg_fs']} Hz")
        print(f"  NIRS HbO shape: {data['nirs_hbo'].shape}, fs={data['nirs_fs']} Hz")
        print(f"  Trials: {len(data['labels'])}, class_names: {data['class_names']}")
        print(f"  Label dist: VF={np.sum(data['labels']==1)}, BL={np.sum(data['labels']==0)}")

        data['eeg'] = bandpass_eeg(data['eeg'], data['eeg_fs'])
        data['nirs_hbo'] = lowpass_nirs(data['nirs_hbo'], data['nirs_fs'])
        data['nirs_hbr'] = lowpass_nirs(data['nirs_hbr'], data['nirs_fs'])

        eeg_wins, nirs_wins, labels = extract_windows(data, task='vf', window_sec=5, step_sec=1)
        all_data[subj] = {'eeg': eeg_wins, 'nirs': nirs_wins, 'labels': labels}
        print(f'  EEG windows: {eeg_wins.shape}')
        print(f'  NIRS windows: {nirs_wins.shape}')
        print(f"  Labels: VF={np.sum(labels==1)}, BL={np.sum(labels==0)}, total={len(labels)}")
    except Exception as e:
        import traceback
        print(f'  ERROR: {e}')
        traceback.print_exc()

print('\nPipeline complete.')


--- VP001 ---


  EEG shape: (371283, 30), fs=200.0 Hz
  NIRS HbO shape: (20107, 36), fs=10.0 Hz
  Trials: 60, class_names: ['VF', 'BL']
  Label dist: VF=30, BL=30


  EEG windows: (360, 30, 1000)
  NIRS windows: (360, 72, 50)
  Labels: VF=180, BL=180, total=360

--- VP002 ---


  EEG shape: (371416, 30), fs=200.0 Hz
  NIRS HbO shape: (20219, 36), fs=10.0 Hz
  Trials: 60, class_names: ['VF', 'BL']
  Label dist: VF=30, BL=30


  EEG windows: (360, 30, 1000)
  NIRS windows: (360, 72, 50)
  Labels: VF=180, BL=180, total=360

--- VP003 ---


...

  EEG windows: (360, 30, 1000)
  NIRS windows: (360, 72, 50)
  Labels: VF=180, BL=180, total=360

--- VP004 ---


  EEG shape: (371949, 30), fs=200.0 Hz
  NIRS HbO shape: (20115, 36), fs=10.0 Hz
  Trials: 60, class_names: ['VF', 'BL']
  Label dist: VF=30, BL=30


  EEG windows: (360, 30, 1000)
  NIRS windows: (360, 72, 50)
  Labels: VF=180, BL=180, total=360

--- VP005 ---


  EEG shape: (372647, 30), fs=200.0 Hz
  NIRS HbO shape: (20237, 36), fs=10.0 Hz
  Trials: 60, class_names: ['VF', 'BL']
  Label dist: VF=30, BL=30


  EEG windows: (360, 30, 1000)
  NIRS windows: (360, 72, 50)
  Labels: VF=180, BL=180, total=360

Pipeline complete.


## 6. Save Processed Data

In [7]:
out_path = '../data/processed/dataset_vf_windows.pkl'
with open(out_path, 'wb') as f:
    pickle.dump(all_data, f)
print(f'Saved to {out_path}')

# Summary
print('\n=== SUMMARY ===')
for subj, d in all_data.items():
    print(f"{subj}: EEG={d['eeg'].shape}, NIRS={d['nirs'].shape}, "
          f"VF={np.sum(d['labels']==1)}, BL={np.sum(d['labels']==0)}, "
          f"total={len(d['labels'])}")


Saved to ../data/processed/dataset_vf_windows.pkl

=== SUMMARY ===
VP001: EEG=(360, 30, 1000), NIRS=(360, 72, 50), VF=180, BL=180, total=360
VP002: EEG=(360, 30, 1000), NIRS=(360, 72, 50), VF=180, BL=180, total=360
VP003: EEG=(360, 30, 1000), NIRS=(360, 72, 50), VF=180, BL=180, total=360
VP004: EEG=(360, 30, 1000), NIRS=(360, 72, 50), VF=180, BL=180, total=360
VP005: EEG=(360, 30, 1000), NIRS=(360, 72, 50), VF=180, BL=180, total=360


## 7. Verification -- Shapes and Label Balance

In [8]:
print('Shape verification:')
for subj, d in all_data.items():
    eeg_ok = d['eeg'].ndim == 3 and d['eeg'].shape[1] == 30 and d['eeg'].shape[2] == 1000
    nirs_ok = d['nirs'].ndim == 3 and d['nirs'].shape[1] == 72 and d['nirs'].shape[2] == 50
    status = 'OK' if (eeg_ok and nirs_ok) else 'FAIL'
    vf_pct = 100 * np.sum(d['labels']==1) / len(d['labels'])
    print(f"  {subj} [{status}]: EEG={d['eeg'].shape}, NIRS={d['nirs'].shape}, "
          f"VF%={vf_pct:.1f}%")

Shape verification:
  VP001 [OK]: EEG=(360, 30, 1000), NIRS=(360, 72, 50), VF%=50.0%
  VP002 [OK]: EEG=(360, 30, 1000), NIRS=(360, 72, 50), VF%=50.0%
  VP003 [OK]: EEG=(360, 30, 1000), NIRS=(360, 72, 50), VF%=50.0%
  VP004 [OK]: EEG=(360, 30, 1000), NIRS=(360, 72, 50), VF%=50.0%
  VP005 [OK]: EEG=(360, 30, 1000), NIRS=(360, 72, 50), VF%=50.0%


## 8. Sanity Check -- Plot Sample Windows from VP001

In [9]:
os.makedirs('../docs/plots', exist_ok=True)

if 'VP001' in all_data:
    d = all_data['VP001']

    # Find first VF window
    vf_idx = np.where(d['labels'] == 1)[0]
    idx = vf_idx[0] if len(vf_idx) > 0 else 0

    eeg_sample = d['eeg'][idx]    # (30, 2000)
    nirs_sample = d['nirs'][idx]  # (72, 100)

    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    lbl_str = 'VF' if d['labels'][idx]==1 else 'BL'
    fig.suptitle(f'VP001 -- Preprocessed Sample (label={lbl_str})',
                 fontsize=13, fontweight='bold')

    # EEG: first 8 channels
    ax = axes[0]
    t_eeg = np.arange(eeg_sample.shape[1]) / 200.0
    offset = np.arange(8) * 50  # uV offset per channel
    for ch in range(8):
        ax.plot(t_eeg, eeg_sample[ch] + offset[ch], linewidth=0.5)
    ax.set_title('EEG -- First 8 channels (bandpass 1-40 Hz)', fontsize=11)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude (uV + offset)')
    ax.set_xlim([0, 10])

    # NIRS HbO: first 6 channels
    ax = axes[1]
    t_nirs = np.arange(nirs_sample.shape[1]) / 10.0
    hbo_sample = nirs_sample[:36, :]  # first 36 = HbO
    for ch in range(6):
        ax.plot(t_nirs, hbo_sample[ch], linewidth=1.2, label=f'Ch{ch+1}')
    ax.set_title('NIRS HbO -- First 6 channels (lowpass 0.2 Hz)', fontsize=11)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Concentration')
    ax.legend(loc='upper right', fontsize=7, ncol=3)
    ax.set_xlim([0, 10])

    # NIRS HbR: first 6 channels
    ax = axes[2]
    hbr_sample = nirs_sample[36:, :]  # last 36 = HbR
    for ch in range(6):
        ax.plot(t_nirs, hbr_sample[ch], linewidth=1.2, label=f'Ch{ch+1}')
    ax.set_title('NIRS HbR -- First 6 channels (lowpass 0.2 Hz)', fontsize=11)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Concentration')
    ax.legend(loc='upper right', fontsize=7, ncol=3)
    ax.set_xlim([0, 10])

    plt.tight_layout()
    out_fig = '../docs/plots/VP001_preprocessed_sample.png'
    plt.savefig(out_fig, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Plot saved to {out_fig}')
else:
    print('VP001 not available in all_data')


Plot saved to ../docs/plots/VP001_preprocessed_sample.png
